In [1]:
!pip install "numpy<2.0" ultralytics filterpy google-generativeai

import numpy as np
import torch
import ultralytics
import filterpy

print(f"Numpy Version: {np.__version__}") 
# If this says 1.26.x, you are safe. 
# If it says 2.x.x, restart the session and run this cell again.

print(f"Ultralytics Version: {ultralytics.__version__}")
print("Systems Check: GREEN.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 30.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.8 MB/s eta 0:00:0000:01m0:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.0 MB/s eta 0:00:00:00:0100:01


In [4]:
import time
import os
import glob
import numpy as np
import cv2
import sqlite3
import faiss
import google.generativeai as genai
from tqdm import tqdm
from collections import deque
import matplotlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from timm.models.vision_transformer import Attention

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

**VectorDB**

In [6]:
# ==============================================================================
# A3F-ViT: TOPOLOGY-AWARE NEURO-SYMBOLIC CORE (WITH HUMAN INTELLIGENCE + RAG/LLM)
# "FINAL DEFENSE EDITION" - COGNITIVE UPGRADE
# ==============================================================================

import os
import glob
import numpy as np
import cv2
import sqlite3 # [NEW] SQL Integration
import faiss   # [NEW] VectorDB Integration
import google.generativeai as genai # [NEW] LLM
from tqdm import tqdm
from collections import deque
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from timm.models.vision_transformer import Attention

# --- SYSTEM MODULES ---
try:
    from ultralytics import YOLO
    from filterpy.kalman import KalmanFilter
    HAS_PHYSICS = True
except ImportError:
    print("[WARN] YOLO or FilterPy not found. Physics module will be disabled.")
    HAS_PHYSICS = False

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
# ==============================================================================
# 1. CONFIGURATION (Fixed Secret Name)
# ==============================================================================
class Config:
    exp_name = "A4F_ViT_Cognitive"
    out_dir = "/kaggle/working/"
    test_folder = "/kaggle/input/datasets/amankumarpatell/drone-anomaly/Bike Roundabout/sequence1/test/04"
    ckpt_path   = "/kaggle/input/datasets/amankumarpatell/da-br-path/best_da_vit_BR.pth"
    
    video_fps = 15.0
    canvas_size = (1280, 720)
    
    # [FIX] Updated to match your secret name "AAAW_SECRET_KEY"
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        # Changed from "GEMINI_API_KEY" to "AAAW_SECRET_KEY"
        gemini_api_key = user_secrets.get_secret("AAAW_SECRET_KEY") 
        genai.configure(api_key=gemini_api_key)
        print("[SUCCESS] Gemini API Key loaded.")
        HAS_LLM = True
    except Exception as e:
        print(f"[WARN] No API Key found ({e}). LLM will run in MOCK mode.")
        HAS_LLM = False
        
    image_size = 224
    num_frames = 4
    planner_trigger_score = 0.60 

# ==============================================================================
# 2. MODEL DEFINITION (Fixed to match Checkpoint)
# ==============================================================================
class RadialMaskMLP(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        # [FIX] Added an extra layer to match the saved 'visual_backbone.pth'
        # The checkpoint has layers up to index 6, so we need 3 hidden layers.
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.ReLU(),          # Layers 0, 1
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), # Layers 2, 3
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), # Layers 4, 5 (Added this)
            nn.Linear(hidden_dim, 1)                      # Layer 6
        )
    def forward(self, r):
        w = F.softplus(self.net(r.view(-1, 1)))
        return w.view(r.shape) / (w.max() + 1e-6)

class AttentionWithMap(Attention):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_map = None
    def forward(self, x, attn_mask=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        self.attn_map = attn 
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class A2F_ViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model('vit_base_patch16_224', pretrained=False)
        for blk in self.encoder.blocks:
            old = blk.attn
            new_attn = AttentionWithMap(dim=768, num_heads=old.num_heads, qkv_bias=True)
            blk.attn = new_attn
        self.decoder = nn.Sequential(
            nn.Conv2d(768 * 4, 512, 1),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64, 3, 3, 2, 1, 1), nn.Tanh()
        )
        self.radial_mlp = RadialMaskMLP()

    def forward(self, x):
        B, T = x.shape[:2]
        feats = []
        for i in range(T):
            f = self.encoder.forward_features(x[:, i])[:, 1:]
            f = f.permute(0, 2, 1).reshape(B, -1, 14, 14)
            feats.append(f)
        return self.decoder(torch.cat(feats, 1))

    def extract_features(self, x):
        """Extracts latent vectors for RAG."""
        with torch.no_grad():
            features = self.encoder.forward_features(x[:, -1])[:, 0] # CLS token
        return features.cpu().numpy()

# ==============================================================================
# 3. COGNITIVE MEMORY (SQL + VectorDB) [FIXED]
# ==============================================================================
class CognitiveMemory:
    def __init__(self, db_path="mission_logs.db"):
        # 1. Initialize SQL (Structured Logs)
        self.conn = sqlite3.connect(db_path)
        self.cursor = self.conn.cursor()
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS events (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp REAL,
                event_type TEXT,
                description TEXT,
                physics_score REAL,
                visual_score REAL
            )
        ''')
        self.conn.commit()

        # 2. Initialize VectorDB (Visual Memory)
        self.dimension = 768 # ViT-Base dimension
        self.index = faiss.IndexFlatL2(self.dimension)
        self.memory_map = {} # Maps VectorID -> Event Description
        self.vector_count = 0

    def add_memory(self, embedding, description, scores, timestamp):
        # Add to VectorDB
        if embedding is not None:
            # Normalize for better cosine similarity approximation with L2
            faiss.normalize_L2(embedding)
            self.index.add(embedding)
            self.memory_map[self.vector_count] = description
            self.vector_count += 1

        # Add to SQL
        # [FIX] Changed 'phy' -> 'physics' and 'vis' -> 'visual' to match 'ctx' keys
        self.cursor.execute('''
            INSERT INTO events (timestamp, event_type, description, physics_score, visual_score)
            VALUES (?, ?, ?, ?, ?)
        ''', (timestamp, "ANOMALY", description, scores['physics'], scores['visual']))
        self.conn.commit()

    def rag_search(self, query_embedding, k=1):
        """Retrieves similar past events based on visual similarity."""
        if self.vector_count == 0: return "No past memory."
        
        faiss.normalize_L2(query_embedding)
        D, I = self.index.search(query_embedding, k)
        
        idx = I[0][0]
        if idx != -1 and idx in self.memory_map:
            return f"Similar to past event: {self.memory_map[idx]}"
        return "No relevant past events found."
# ==============================================================================
# 4. LLM AGENTIC PLANNER [NEW]
# ==============================================================================
class LLMPlanner:
    def __init__(self):
        self.model = genai.GenerativeModel('gemini-pro') if Config.HAS_LLM else None
        self.last_call = 0
        self.cooldown = 5.0 # Seconds between LLM calls to save quota

    def generate_plan(self, ctx, mood, rag_context):
        # Rate Limiting
        if time.time() - self.last_call < self.cooldown:
            return self._fallback_plan(ctx, mood)
        
        if not self.model:
            return self._fallback_plan(ctx, mood)

        prompt = f"""
        You are an Autonomous Drone AI. Analyze this telemetry:
        - Visual Anomaly Score: {ctx['visual']:.2f} (0-1)
        - Physics Anomaly Score: {ctx['physics']:.2f} (0-1)
        - Driver Intent: {mood}
        - Historical Context (RAG): {rag_context}
        
        Based on this, output a JSON with:
        1. 'action': One word (INTERCEPT, TRACK, PATROL, REDIRECT).
        2. 'reasoning': Max 10 words explanation.
        """
        
        try:
            response = self.model.generate_content(prompt)
            # Simple parsing (In prod, use strict JSON parsing)
            text = response.text
            import json
            # Heuristic to find JSON in response
            start = text.find('{')
            end = text.rfind('}') + 1
            data = json.loads(text[start:end])
            self.last_call = time.time()
            return data
        except:
            return self._fallback_plan(ctx, mood)

    def _fallback_plan(self, ctx, mood):
        # The logic from your old code
        f = ctx['fused']
        if ctx['topo']: return {"action": "REDIRECT", "reasoning": "Terrain Violation."}
        if f > 0.8: return {"action": "INTERCEPT", "reasoning": f"Hybrid Lock: {mood}."}
        elif ctx['physics'] > 0.6: return {"action": "TRACK", "reasoning": "Physics deviation."}
        else: return {"action": "PATROL", "reasoning": "Sector clear."}

# ==============================================================================
# 5. PHYSICS & HUMAN BEHAVIOR (Kept from your code)
# ==============================================================================
class HumanBehaviorEngine:
    def __init__(self):
        self.vel_history = deque(maxlen=10)
    
    def analyze_intent(self, velocity_vector):
        self.vel_history.append(velocity_vector)
        if len(self.vel_history) < 3: return "ANALYZING", 0.0
        
        accels = []
        for i in range(1, len(self.vel_history)):
            v1 = np.array(self.vel_history[i-1]); v2 = np.array(self.vel_history[i])
            accels.append(np.linalg.norm(v2 - v1))
            
        if len(accels) < 2: return "ANALYZING", 0.0
        jerks = np.diff(accels)
        avg_jerk = np.mean(np.abs(jerks)) * 100.0
        intent_score = min(1.0, avg_jerk / 5.0)
        
        if intent_score > 0.8: return "AGGRESSIVE", intent_score
        elif intent_score > 0.4: return "DISTRACTED", intent_score
        elif intent_score < 0.1: return "CAUTIOUS", intent_score
        else: return "NORMAL", intent_score

class PhysicsOracle:
    def __init__(self):
        if HAS_PHYSICS:
            try: self.model = YOLO('yolov8n.pt')
            except: self.model = None
            self.kf = KalmanFilter(dim_x=4, dim_z=2)
            self.kf.F = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]])
            self.kf.H = np.array([[1,0,0,0],[0,1,0,0]])
            self.kf.P *= 1000.; self.kf.R *= 5.; self.kf.Q *= 5.0
            self.kf.x = np.zeros(4)
            
    def get_state(self, frame_path):
        if not HAS_PHYSICS or self.model is None: return None, 0.0, False
        results = self.model(frame_path, classes=[2,3,5,7], verbose=False)
        bbox_norm = None; score = 0.0; topo_alert = False 
        
        if len(results[0].boxes) > 0:
            b = results[0].boxes.xywh.cpu().numpy()[0]
            H, W = results[0].orig_shape
            bbox_norm = (b[0]/W, b[1]/H, b[2]/W, b[3]/H) 
            
            dist_from_center = np.sqrt((bbox_norm[0]-0.5)**2 + (bbox_norm[1]-0.5)**2)
            if dist_from_center < 0.12 or dist_from_center > 0.48: topo_alert = True
            
            z = np.array([bbox_norm[0], bbox_norm[1]])
            self.kf.predict(); self.kf.update(z)
            pred = self.kf.H @ self.kf.x_prior
            score = np.linalg.norm(z - pred) * 10.0 
            
        return bbox_norm, min(1.0, score), topo_alert

# ==============================================================================
# 6. VISUALIZATION (Dashboard)
# ==============================================================================
class ResearchDashboard:
    def __init__(self, save_path):
        self.save_path = save_path
        self.writer = None
        self.log_trace = deque(["System Init...", "SQL Connected...", "VectorDB Ready."], maxlen=8)
        self.c_bg = (10, 15, 20); self.c_panel = (30, 35, 40)
        self.c_vis = (0, 165, 255); self.c_phy = (255, 200, 0)
        self.c_crit = (0, 0, 255); self.c_txt = (220, 220, 220)
        
    def init_writer(self):
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        self.writer = cv2.VideoWriter(self.save_path, fourcc, Config.video_fps, Config.canvas_size)

    def update(self, frame_rgb, bbox_norm, state, plan, mood):
        canvas = np.zeros((Config.canvas_size[1], Config.canvas_size[0], 3), dtype=np.uint8)
        canvas[:] = self.c_bg
        
        # Log Management
        if plan and plan.get('reasoning') and (not self.log_trace or plan['reasoning'] != self.log_trace[-1]):
            self.log_trace.append(f"> {plan['action']}: {plan['reasoning']}")

        # Video Feed
        feed = cv2.resize(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR), (720, 720))
        
        # Draw Target Box
        if bbox_norm:
            h, w = 720, 720
            cx, cy = int(bbox_norm[0]*w), int(bbox_norm[1]*h)
            bw, bh = int(bbox_norm[2]*w), int(bbox_norm[3]*h)
            color = self.c_crit if state['alert'] else (0, 255, 0)
            cv2.rectangle(feed, (cx-bw//2, cy-bh//2), (cx+bw//2, cy+bh//2), color, 2)
            cv2.putText(feed, f"{mood} | P:{state['phy']:.2f}", (cx-bw//2, cy-bh//2-10), cv2.FONT_HERSHEY_PLAIN, 1, color, 1)

        canvas[0:720, 0:720] = feed
        
        # Panel Info
        px = 740
        cv2.putText(canvas, "COGNITIVE CORE", (px, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        cv2.putText(canvas, f"LLM STATUS: {'ONLINE' if Config.HAS_LLM else 'OFFLINE'}", (px, 70), cv2.FONT_HERSHEY_PLAIN, 1, (0,255,0), 1)
        
        # Logs
        for i, log in enumerate(self.log_trace):
            col = self.c_txt
            if "INTERCEPT" in log: col = self.c_crit
            cv2.putText(canvas, log, (px, 120 + (i*25)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
            
        if self.writer: self.writer.write(canvas)

    def close(self):
        if self.writer: self.writer.release()

# ==============================================================================
# 7. MAIN EXECUTION
# ==============================================================================
def main():
    cfg = Config()
    os.makedirs(cfg.out_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Init Cognitive Modules [NEW]
    memory = CognitiveMemory(os.path.join(cfg.out_dir, "blackbox.db"))
    planner = LLMPlanner()
    
    # 2. Init Visual & Physics
    model = A2F_ViT().to(device)
    if os.path.exists(cfg.ckpt_path):
        model.load_state_dict(torch.load(cfg.ckpt_path, map_location=device))
    model.eval()
    
    oracle = PhysicsOracle()
    human_brain = HumanBehaviorEngine()
    dash = ResearchDashboard(os.path.join(cfg.out_dir, "Cognitive_Demo.mp4"))
    dash.init_writer()
    
    # Load Data
    frames = sorted(glob.glob(os.path.join(cfg.test_folder, "**/*.jpg"), recursive=True))
    if not frames: print("No frames found."); return
    
    print(f"[INFO] Running Cognitive Pipeline on {len(frames)} frames...")
    
    for i in tqdm(range(0, len(frames)-Config.num_frames)):
        clip_paths = frames[i:i+Config.num_frames]
        target_path = frames[i+Config.num_frames]
        
        # Load Images
        clip = []
        for p in clip_paths:
            img = cv2.imread(p)
            if img is None: continue
            img = cv2.resize(img, (224,224)).astype(np.float32)/127.5-1.0
            clip.append(torch.from_numpy(img).permute(2,0,1))
        if len(clip) < Config.num_frames: continue
        
        inp = torch.stack(clip).unsqueeze(0).to(device)
        img_raw = cv2.imread(target_path)
        img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
        
        # --- 1. SENSING (Visual + Physics) ---
        with torch.no_grad():
            pred = model(inp) # Reconstruction
            # [NEW] Extract Embedding for VectorDB
            embedding = model.extract_features(inp) 
            
            # Simple Visual Score Calculation
            t_img = cv2.resize(img_raw, (224,224)).astype(np.float32)/127.5-1.0
            t_ts = torch.from_numpy(t_img).permute(2,0,1).unsqueeze(0).to(device)
            raw_err = torch.abs(t_ts - pred).reshape(-1)
            topk, _ = torch.topk(raw_err, k=1000)
            vis_score = min(1.0, topk.mean().item() * 5.0)

        bbox_norm, phy_score, topo_alert = oracle.get_state(target_path)
        
        driver_mood = "NORMAL"
        intent_score = 0.0
        if bbox_norm:
            driver_mood, intent_score = human_brain.analyze_intent((oracle.kf.x[2], oracle.kf.x[3]))

        # Fusion
        fused = (0.5 * vis_score) + (0.3 * phy_score) + (0.2 * intent_score)
        is_alert = (fused > Config.planner_trigger_score) or topo_alert

        # --- 2. THINKING (RAG + LLM) [NEW] ---
        rag_context = "No Context"
        plan = None
        
        if is_alert:
            # A. Retrieve similar past events (RAG)
            rag_context = memory.rag_search(embedding)
            
            # B. Ask LLM for plan
            ctx = {'visual': vis_score, 'physics': phy_score, 'fused': fused, 'topo': topo_alert}
            plan = planner.generate_plan(ctx, driver_mood, rag_context)
            
            # C. Save to Memory (SQL + VectorDB)
            memory.add_memory(embedding, plan['reasoning'], ctx, time.time())

        # Update Dash
        state = {'vis': vis_score, 'phy': phy_score, 'alert': is_alert}
        dash.update(img_rgb, bbox_norm, state, plan, driver_mood)

    dash.close()
    print(f"[SUCCESS] Video saved. SQL Logs stored at {os.path.join(cfg.out_dir, 'mission_logs.db')}")

if __name__ == "__main__":
    main()

[SUCCESS] Gemini API Key loaded.
[INFO] Running Cognitive Pipeline on 5700 frames...


100%|██████████| 5696/5696 [12:23<00:00,  7.66it/s]

[SUCCESS] Video saved. SQL Logs stored at /kaggle/working/mission_logs.db


In [7]:
import sqlite3
import pandas as pd
import faiss

# 1. Inspect SQL Logs (The "Flight Recorder")
db_path = "/kaggle/working/blackbox.db"
if os.path.exists(db_path):
    print(f"\n[INFO] Reading Mission Logs from: {db_path}")
    conn = sqlite3.connect(db_path)
    # Load last 5 events into a Pandas DataFrame for nice display
    df = pd.read_sql_query("SELECT * FROM events ORDER BY id DESC LIMIT 5", conn)
    print(df.to_string())
    conn.close()
else:
    print("[FAIL] No SQL Database found. Did the main loop run?")

# 2. Inspect VectorDB (The "Visual Memory")
# Note: We didn't save the index to disk in the simple script, 
# but we can check if it's in memory if you run this in the same session.
# If you want to verify it populated, check the log file output in the video
# or check if the 'df' above has entries (since SQL and VectorDB update together).
print(f"\n[INFO] Structured Event Logs & Embeddings are active.")


[INFO] Reading Mission Logs from: /kaggle/working/blackbox.db
    id     timestamp event_type               description  physics_score  visual_score
0  160  1.775740e+09    ANOMALY             Sector clear.       0.180382           1.0
1  159  1.775740e+09    ANOMALY  Hybrid Lock: AGGRESSIVE.       0.640382           1.0
2  158  1.775740e+09    ANOMALY  Hybrid Lock: AGGRESSIVE.       1.000000           1.0
3  157  1.775739e+09    ANOMALY  Hybrid Lock: AGGRESSIVE.       1.000000           1.0
4  156  1.775739e+09    ANOMALY        Terrain Violation.       0.986492           1.0

[INFO] Structured Event Logs & Embeddings are active.


MTP_Final_Video

In [6]:
# ==============================================================================
# A3F-ViT: TOPOLOGY-AWARE NEURO-SYMBOLIC CORE (WITH HUMAN INTELLIGENCE)
# "FINAL DEFENSE EDITION" - DEC 4TH TARGET
# Includes: Spectral Visual, Kalman Physics, Human Behavior (Jerk), Geofence, & 3D Report
# ==============================================================================

import os
import glob
import numpy as np
import cv2
from tqdm import tqdm
from collections import deque
import matplotlib
matplotlib.use('Agg') # Backend for saving plots without display
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from timm.models.vision_transformer import Attention

# --- SYSTEM MODULES ---
try:
    from ultralytics import YOLO
    from filterpy.kalman import KalmanFilter
    HAS_PHYSICS = True
except ImportError:
    print("[WARN] YOLO or FilterPy not found. Physics module will be disabled.")
    HAS_PHYSICS = False

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
class Config:
    exp_name = "A4F_ViT_Human"
    out_dir = "/kaggle/working/"
    # UPDATE THESE PATHS BEFORE RUNNING
    test_folder = "/kaggle/input/datasets/amankumarpatell/drone-anomaly/Bike Roundabout/sequence1/test/04"
    ckpt_path   = "/kaggle/input/datasets/amankumarpatell/da-br-path/best_da_vit_BR.pth"
    
    video_fps = 15.0
    canvas_size = (1280, 720)
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        gemini_api_key = user_secrets.get_secret("AAAW_SECRET_KEY")
        print("[SUCCESS] Gemini API Key loaded from Secrets.")
    except Exception as e:
        print(f"[WARN] Could not load API Key: {e}. Using Mock.")
        gemini_api_key = "MOCK_KEY"
        
    image_size = 224
    num_frames = 4
    planner_trigger_score = 0.60 

# ==============================================================================
# 2. MODEL DEFINITION (Visual Cortex)
# ==============================================================================
class RadialMaskMLP(nn.Module):
    def __init__(self, hidden_dim=32, layers=3):
        super().__init__()
        seq = [nn.Linear(1, hidden_dim), nn.ReLU()]
        for _ in range(layers-1):
            seq.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        seq.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*seq)
    def forward(self, r):
        w = F.softplus(self.net(r.view(-1, 1)))
        return w.view(r.shape) / (w.max() + 1e-6)

class AttentionWithMap(Attention):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_map = None
    def forward(self, x, attn_mask=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        self.attn_map = attn 
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class A2F_ViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model('vit_base_patch16_224', pretrained=False)
        for blk in self.encoder.blocks:
            old = blk.attn
            new_attn = AttentionWithMap(dim=768, num_heads=old.num_heads, qkv_bias=True)
            blk.attn = new_attn
        self.decoder = nn.Sequential(
            nn.Conv2d(768 * 4, 512, 1),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64, 3, 3, 2, 1, 1), nn.Tanh()
        )
        self.radial_mlp = RadialMaskMLP()

    def forward(self, x):
        B, T = x.shape[:2]
        feats = []
        for i in range(T):
            f = self.encoder.forward_features(x[:, i])[:, 1:]
            f = f.permute(0, 2, 1).reshape(B, -1, 14, 14)
            feats.append(f)
        return self.decoder(torch.cat(feats, 1))

# ==============================================================================
# 3. HUMAN INTELLIGENCE ENGINE (NEW)
# ==============================================================================
class HumanBehaviorEngine:
    def __init__(self):
        # Store history of velocities to calculate acceleration/jerk
        self.vel_history = deque(maxlen=10)
    
    def analyze_intent(self, velocity_vector):
        """
        Infers driver state based on motion smoothness (Jerk).
        Input: velocity_vector (vx, vy)
        Returns: State String, Intent Score (0.0 - 1.0)
        """
        self.vel_history.append(velocity_vector)
        
        # Need at least 3 frames to calculate Jerk (Change in Accel)
        if len(self.vel_history) < 3:
            return "ANALYZING", 0.0

        # 1. Calculate Acceleration (Delta V)
        accels = []
        for i in range(1, len(self.vel_history)):
            v1 = np.array(self.vel_history[i-1])
            v2 = np.array(self.vel_history[i])
            accels.append(np.linalg.norm(v2 - v1))
            
        # 2. Calculate Jerk (Delta A) - The "Comfort" Metric
        if len(accels) < 2: return "ANALYZING", 0.0
        
        jerks = np.diff(accels)
        avg_jerk = np.mean(np.abs(jerks)) * 100.0 # Scale up for readability
        
        # 3. Classify "Human State"
        intent_score = min(1.0, avg_jerk / 5.0) # Normalize 0-1
        
        if intent_score > 0.8: return "AGGRESSIVE", intent_score
        elif intent_score > 0.4: return "DISTRACTED", intent_score
        elif intent_score < 0.1: return "CAUTIOUS", intent_score
        else: return "NORMAL", intent_score

# ==============================================================================
# 4. PHYSICS ORACLE
# ==============================================================================
class PhysicsOracle:
    def __init__(self):
        if HAS_PHYSICS:
            try: self.model = YOLO('yolov8n.pt')
            except: self.model = None
            
            # Kalman Filter Setup
            self.kf = KalmanFilter(dim_x=4, dim_z=2) # [x, y, vx, vy]
            self.kf.F = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]])
            self.kf.H = np.array([[1,0,0,0],[0,1,0,0]])
            self.kf.P *= 1000.; self.kf.R *= 5.; self.kf.Q *= 5.0
            self.kf.x = np.zeros(4)
            
    def get_state(self, frame_path):
        if not HAS_PHYSICS or self.model is None:
            return None, 0.0, False

        results = self.model(frame_path, classes=[2,3,5,7], verbose=False)
        bbox_norm = None
        score = 0.0
        topo_alert = False 
        
        if len(results[0].boxes) > 0:
            b = results[0].boxes.xywh.cpu().numpy()[0]
            H, W = results[0].orig_shape
            bbox_norm = (b[0]/W, b[1]/H, b[2]/W, b[3]/H) 
            
            # Topology Check
            dist_from_center = np.sqrt((bbox_norm[0]-0.5)**2 + (bbox_norm[1]-0.5)**2)
            if dist_from_center < 0.12 or dist_from_center > 0.48:
                topo_alert = True
            
            # Kalman Update
            z = np.array([bbox_norm[0], bbox_norm[1]])
            self.kf.predict()
            self.kf.update(z)
            pred = self.kf.H @ self.kf.x_prior
            score = np.linalg.norm(z - pred) * 10.0 
            
        return bbox_norm, min(1.0, score), topo_alert

# ==============================================================================
# 5. DASHBOARD & VISUALIZATION
# ==============================================================================
class ResearchDashboard:
    def __init__(self, save_path):
        self.save_path = save_path
        self.writer = None
        self.history_len = 50
        self.vis_trace = deque([0.0]*self.history_len, maxlen=self.history_len)
        self.phy_trace = deque([0.0]*self.history_len, maxlen=self.history_len)
        self.log_trace = deque(["System Initialized...", "Human Engine Active."], maxlen=8)
        self.trajectory = deque(maxlen=20) 
        self.radar_trail = deque(maxlen=20) 
        self.smooth_vis = 0.0
        self.smooth_phy = 0.0
        self.lock_timer = 0  
        
        self.c_bg = (10, 15, 20); self.c_panel = (30, 35, 40)
        self.c_vis = (0, 165, 255); self.c_phy = (255, 200, 0)
        self.c_crit = (0, 0, 255); self.c_scan = (255, 255, 0) 
        self.c_safe = (0, 255, 0); self.c_topo = (255, 0, 255) 
        self.c_txt = (220, 220, 220)
        
    def init_writer(self):
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        self.writer = cv2.VideoWriter(self.save_path, fourcc, Config.video_fps, Config.canvas_size)

    def _draw_phase_radar(self, img, state_vec, pos, size=180):
        # --- ULTIMATE 3D ENGINE (With Depth Sorting) ---
        x0, y0 = pos
        center = np.array([x0 + size//2, y0 + size//2 + 20])
        scale = size // 2.5

        # 1. Input Mapping (-1 to 1)
        val_x = (state_vec['vis'] * 2) - 1
        val_y = (state_vec['phy'] * 2) - 1
        val_z = (state_vec.get('topo', 0) * 2) - 1

        # 2. Camera Angles
        if not hasattr(self, 'rot_angle'): self.rot_angle = 0
        self.rot_angle += 0.015
        
        elevation = np.radians(25) 
        azimuth = np.radians(45) + self.rot_angle

        # Rotation Matrices
        rot_y = np.array([[np.cos(azimuth), 0, np.sin(azimuth)], [0, 1, 0], [-np.sin(azimuth), 0, np.cos(azimuth)]])
        rot_x = np.array([[1, 0, 0], [0, np.cos(elevation), -np.sin(elevation)], [0, np.sin(elevation), np.cos(elevation)]])

        def project(point):
            # point is (x, y, z). We map Input Z -> World Y (Height)
            p = np.array([point[0], point[2], point[1]]) 
            rotated = p @ rot_y @ rot_x
            
            z_depth = 3.0 - rotated[2] # Depth from camera
            factor = scale / z_depth
            
            px = int(center[0] + rotated[0] * factor)
            py = int(center[1] - rotated[1] * factor)
            return (px, py), z_depth # Return depth too!

        # 3. Define Faces (Not just lines)
        # We define the 6 faces of the cube to sort them
        # (Bottom, Top, Back, Front, Left, Right)
        # Order: bl, br, tr, tl
        faces = [
            {"pts": [(-1,-1,-1), (1,-1,-1), (1,1,-1), (-1,1,-1)], "color": (30,30,30), "id": "floor"}, # Floor
            {"pts": [(-1,-1,1), (1,-1,1), (1,1,1), (-1,1,1)], "color": (20,20,20), "id": "ceil"},   # Ceiling
            {"pts": [(-1,-1,-1), (1,-1,-1), (1,-1,1), (-1,-1,1)], "color": (20,20,20), "id": "back"}, # Back
            {"pts": [(-1,1,-1), (1,1,-1), (1,1,1), (-1,1,1)], "color": (20,20,20), "id": "front"},  # Front
            {"pts": [(-1,-1,-1), (-1,1,-1), (-1,1,1), (-1,-1,1)], "color": (20,20,20), "id": "left"}, # Left
            {"pts": [(1,-1,-1), (1,1,-1), (1,1,1), (1,-1,1)], "color": (20,20,20), "id": "right"}   # Right
        ]

        # 4. Calculate Average Depth for each Face
        render_list = []
        for face in faces:
            proj_pts = []
            avg_z = 0
            for pt in face["pts"]:
                p2d, z = project(pt)
                proj_pts.append(p2d)
                avg_z += z
            avg_z /= 4.0
            render_list.append((avg_z, face, proj_pts))

        # 5. SORT Faces (Furthest first -> "Painter's Algorithm")
        render_list.sort(key=lambda x: x[0], reverse=True)

        # 6. Render Sorted Faces
        for _, face, pts in render_list:
            # Draw faint grid on the floor ONLY
            if face["id"] == "floor":
                # Draw grid lines
                cv2.line(img, pts[0], pts[2], (40,40,40), 1)
                cv2.line(img, pts[1], pts[3], (40,40,40), 1)
                
            # Draw Edges of this face (Wireframe)
            # Only draw "Back" edges dim, "Front" edges bright? 
            # Actually, just drawing sorted edges solves the overlap issue.
            for i in range(4):
                cv2.line(img, pts[i], pts[(i+1)%4], (50, 50, 50), 1)

        # 7. Re-Draw Main Axes ON TOP (so they are never hidden)
        origin, _ = project((-1,-1,-1))
        px, _ = project((1,-1,-1))
        py, _ = project((-1,1,-1)) # Depth Y
        pz, _ = project((-1,-1,1)) # Height Z
        
        cv2.line(img, origin, px, self.c_vis, 2)
        cv2.line(img, origin, py, self.c_phy, 2)
        cv2.line(img, origin, pz, self.c_topo, 2)

        # Labels
        cv2.putText(img, "VIS", px, cv2.FONT_HERSHEY_SIMPLEX, 0.35, self.c_vis, 1)
        cv2.putText(img, "PHY", py, cv2.FONT_HERSHEY_SIMPLEX, 0.35, self.c_phy, 1)
        cv2.putText(img, "RISK", pz, cv2.FONT_HERSHEY_SIMPLEX, 0.35, self.c_topo, 1)

        # 8. The Dot & Trail (Always drawn last = Always on top)
        curr_pos = (val_x, val_y, val_z)
        p_curr, _ = project(curr_pos)
        
        # Trail
        self.radar_trail.append(p_curr)
        if len(self.radar_trail) > 15: self.radar_trail.popleft()
        
        for i in range(len(self.radar_trail)):
            alpha = (i+1)/len(self.radar_trail)
            color = int(255*alpha)
            cv2.circle(img, self.radar_trail[i], 1, (color,color,color), -1)

        # Drop Line
        floor_pt, _ = project((val_x, val_y, -1))
        cv2.line(img, p_curr, floor_pt, (100,100,100), 1)
        cv2.circle(img, floor_pt, 2, (50,50,50), -1)

        # Hero Dot
        col = self.c_scan
        if state_vec['alert']: col = self.c_crit
        elif state_vec.get('topo', 0) > 0.5: col = self.c_topo
        elif state_vec['phy'] > 0.5: col = self.c_phy
        
        cv2.circle(img, p_curr, 5, col, -1)
        cv2.circle(img, p_curr, 2, (255,255,255), -1)
        
    def _draw_graph(self, img, roi, data, color, label):
        x0, y0, w, h = roi
        cv2.rectangle(img, (x0, y0), (x0+w, y0+h), self.c_panel, -1)
        cv2.putText(img, label, (x0+10, y0+20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, self.c_txt, 1)
        if len(data) < 2: return
        pts = []
        max_val = max(1.0, max(data))
        for i, val in enumerate(data):
            px = int(x0 + (i / (self.history_len-1)) * w)
            py = int(y0 + h - (val / max_val) * (h-10) - 5)
            pts.append((px, py))
        for i in range(1, len(pts)): cv2.line(img, pts[i-1], pts[i], color, 2)
        cv2.putText(img, f"{data[-1]:.2f}", (x0+w-50, y0+20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # --- MAIN UPDATE FUNCTION (Defense Grid + Bio-Gaze + PiP) ---
   # --- MAIN UPDATE FUNCTION (Defense Grid + Bio-Gaze + PiP) ---
    def update(self, frame_rgb, bbox_norm, state, plan, mood):
        # 1. Canvas Setup
        canvas = np.zeros((Config.canvas_size[1], Config.canvas_size[0], 3), dtype=np.uint8)
        canvas[:] = self.c_bg
        
        # 2. Smoothing Data
        self.smooth_vis = (0.8 * self.smooth_vis) + (0.2 * state['vis'])
        self.smooth_phy = (0.8 * self.smooth_phy) + (0.2 * state['phy'])
        self.vis_trace.append(self.smooth_vis)
        self.phy_trace.append(self.smooth_phy)
        
        # Instant Log Sync
        if state['alert']:
            current_log = f"> !!! INTERCEPT: {mood} Threat Detected !!!"
            if self.log_trace[-1] != current_log: self.log_trace.append(current_log)
        elif plan and plan.get('reasoning') and plan['reasoning'] != self.log_trace[-1]:
            self.log_trace.append(f"> {plan['action']}: {plan['reasoning'][:35]}...")

        # 3. Video Feed Prep
        feed_size = 720
        feed = cv2.resize(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR), (feed_size, feed_size))
        center_x, center_y = feed_size // 2, feed_size // 2

        # --- PREPARE TARGET COORDS ---
        curr_cx, curr_cy, curr_bw, curr_bh = None, None, 0, 0
        target_active = False
        if bbox_norm:
             nx, ny, nw, nh = bbox_norm
             curr_cx, curr_cy = int(nx*feed_size), int(ny*feed_size)
             curr_bw, curr_bh = int(nw*feed_size), int(nh*feed_size)
             target_active = True

        # --- NOVELTY 1: BIO-GAZE SPOTLIGHT (Apply BEFORE HUD) ---
        # This darkens the edges and highlights the target
        self._draw_bio_gaze(feed, curr_cx, curr_cy, target_active)

        # --- NOVELTY 2: TACTICAL HUD LAYER (Grid & Chevrons) ---
        self._draw_tactical_hud(feed, curr_cx, curr_cy, state)

        # 4. Geofence (Sonar Pulse)
        pulse = int(5 * np.sin(len(self.vis_trace) * 0.15)) 
        ring_color = (60, 40, 60); thickness = 1
        if state['topo']: ring_color = self.c_topo; thickness = 3
        cv2.circle(feed, (center_x, center_y), int(0.12*feed_size*2) + pulse, ring_color, thickness)
        
        # 5. TARGET TRACKING BOX
        if target_active:
            # Trajectory History
            if len(self.trajectory) > 0:
                last_cx, last_cy = self.trajectory[-1]
                if np.sqrt((curr_cx-last_cx)**2 + (curr_cy-last_cy)**2) > 100: self.trajectory.clear()
            self.trajectory.append((curr_cx, curr_cy))
            
            # State Logic
            color = self.c_scan; status = "SCANNING"; 
            if state['alert']: self.lock_timer = 30
            
            if self.lock_timer > 0: 
                color = self.c_crit; status = f"LOCK | {mood}"
                self.lock_timer -= 1
            elif state['topo']: 
                color = self.c_topo; status = "ZONE BREACH"
            
            # Engagement Vector (Drone to Target)
            cv2.line(feed, (center_x, center_y), (curr_cx, curr_cy), color, 1)
            
            # Iron Man Box
            cv2.rectangle(feed, (curr_cx-curr_bw//2, curr_cy-curr_bh//2), (curr_cx+curr_bw//2, curr_cy+curr_bh//2), color, 1)
            len_c = int(curr_bw * 0.25)
            # Corners...
            pts = [((curr_cx-curr_bw//2, curr_cy-curr_bh//2), (1,0), (0,1)), 
                   ((curr_cx+curr_bw//2, curr_cy-curr_bh//2), (-1,0), (0,1)),
                   ((curr_cx-curr_bw//2, curr_cy+curr_bh//2), (1,0), (0,-1)),
                   ((curr_cx+curr_bw//2, curr_cy+curr_bh//2), (-1,0), (0,-1))]
            for pt, dx, dy in pts:
                p1 = (pt[0] + dx[0]*len_c, pt[1] + dx[1]*len_c)
                p2 = (pt[0] + dy[0]*len_c, pt[1] + dy[1]*len_c)
                cv2.line(feed, pt, p1, color, 3)
                cv2.line(feed, pt, p2, color, 3)

            # Telemetry Block (Military Data)
            tx = curr_cx + curr_bw//2 + 8
            ty = curr_cy - curr_bh//2 + 10
            cv2.rectangle(feed, (tx, ty-15), (tx+130, ty+45), (0,0,0), -1)
            cv2.putText(feed, status, (tx, ty), cv2.FONT_HERSHEY_PLAIN, 1.0, color, 1)
            dist_m = int((feed_size - curr_cy) * 0.2) 
            cv2.putText(feed, f"RNG: {dist_m}m", (tx, ty+15), cv2.FONT_HERSHEY_PLAIN, 0.8, (200,200,200), 1)
            cv2.putText(feed, f"BRG: {int((curr_cx/feed_size)*360)}", (tx, ty+30), cv2.FONT_HERSHEY_PLAIN, 0.8, (200,200,200), 1)
            
            # --- NOVELTY 3: PiP TARGET ACQUISITION CAM (Bottom Right) ---
            # Extract ROI
            y1, y2 = max(0, curr_cy - curr_bh), min(feed_size, curr_cy + curr_bh)
            x1, x2 = max(0, curr_cx - curr_bw), min(feed_size, curr_cx + curr_bw)
            
            if (x2-x1) > 10 and (y2-y1) > 10:
                roi = feed[y1:y2, x1:x2]
                pip_size = 150
                try:
                    roi_resized = cv2.resize(roi, (pip_size, pip_size))
                    # Border & Label
                    cv2.rectangle(roi_resized, (0,0), (pip_size-1, pip_size-1), color, 4)
                    cv2.putText(roi_resized, "ZOOM x4", (5, pip_size-5), cv2.FONT_HERSHEY_PLAIN, 0.8, color, 1)
                    # Paste
                    feed[feed_size-pip_size-10:feed_size-10, feed_size-pip_size-10:feed_size-10] = roi_resized
                    # Tether Line
                    cv2.line(feed, (curr_cx+curr_bw//2, curr_cy+curr_bh//2), (feed_size-pip_size-10, feed_size-pip_size-10), color, 1)
                except: pass # Handle edge case where resize fails

        else:
            self.trajectory.clear()
        
        canvas[0:720, 0:720] = feed
        
        # 6. PANEL & GRAPHS (Unified & 3D)
        panel_x = 740
        cv2.putText(canvas, " TACTICAL CORE", (panel_x, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        smooth_state = {
            'vis': self.smooth_vis, 'phy': self.smooth_phy, 
            'alert': (self.lock_timer > 0), 'topo': min(1.0, state['topo'] * 3.0) 
        }
        self._draw_phase_radar(canvas, smooth_state, (panel_x + 150, 60), size=200)
        
        # Graphs (Color Synced)
        self._draw_graph(canvas, (panel_x, 300, 500, 100), self.vis_trace, self.c_vis, "SPECTRAL SIG (VIS)")
        self._draw_graph(canvas, (panel_x, 420, 500, 100), self.phy_trace, self.c_phy, "KINEMATIC DIV (PHY)")
        
        # 7. Logs
        cv2.putText(canvas, f"PROFILE: {mood}", (panel_x+10, 540), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        cv2.rectangle(canvas, (panel_x, 550), (panel_x+500, 700), self.c_panel, -1)
        cv2.putText(canvas, "COMMAND LOG:", (panel_x+10, 575), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)
        
        for i, log in enumerate(self.log_trace):
            col = self.c_txt
            if "INTERCEPT" in log: col = self.c_crit
            elif "Violation" in log: col = self.c_topo 
            cv2.putText(canvas, log, (panel_x+10, 600 + (i*20)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
            
        if self.writer: self.writer.write(canvas)

    
    def _draw_tactical_hud(self, img, cx, cy, state):
        h, w = img.shape[:2]
        
        # 1. Hex-Grid / MGRS (Dark Cyan)
        # Using simple lines for speed, but cleaner color
        grid_col = (40, 40, 0)
        for x in range(0, w, 120):
            cv2.line(img, (x, 0), (x, h), grid_col, 1)
            if x % 240 == 0: cv2.putText(img, f"GRID {10+int(x/10)}", (x+2, h-5), cv2.FONT_HERSHEY_PLAIN, 0.7, grid_col, 1)
        for y in range(0, h, 120):
            cv2.line(img, (0, y), (w, y), grid_col, 1)

        # 2. Central Reticle (Drone Aim Point)
        cv2.drawMarker(img, (w//2, h//2), (200, 200, 200), cv2.MARKER_CROSS, 40, 1)
        
        # 3. Ghost Track (Active Chevrons)
        if cx and cy:
            # Predict future position
            vel_factor = int(40 + (state['phy'] * 250)) 
            if len(self.trajectory) > 2:
                dx = self.trajectory[-1][0] - self.trajectory[-2][0]
                dy = self.trajectory[-1][1] - self.trajectory[-2][1]
                mag = np.sqrt(dx*dx + dy*dy)
                
                if mag > 1: 
                    pred_x = int(cx + (dx/mag) * vel_factor)
                    pred_y = int(cy + (dy/mag) * vel_factor)
                    
                    # Color Logic: Sync with Graph Colors!
                    # Physics (Orange) is the dominant vector force
                    i_col = self.c_phy 
                    if state['alert']: i_col = self.c_crit
                    
                    # Draw Chevrons (>>>) instead of line
                    pts = np.linspace((cx, cy), (pred_x, pred_y), num=6).astype(int)
                    for i in range(1, len(pts)):
                        # Draw Arrowhead
                        cv2.arrowedLine(img, tuple(pts[i-1]), tuple(pts[i]), i_col, 2, tipLength=0.4)
                    
                    # Impact Time Label
                    tts = max(0.1, 2.0 - state['phy']) # Time to impact
                    cv2.putText(img, f"T-IMP: {tts:.1f}s", (pred_x+10, pred_y), cv2.FONT_HERSHEY_PLAIN, 0.8, i_col, 1)

    # --- HELPER: BIO-INSPIRED GAZE (Simulates Human Eye Attention) ---
   # --- HELPER: BIO-INSPIRED GAZE (Simulates Human Eye Attention) ---
    def _draw_bio_gaze(self, img, target_cx, target_cy, active):
        h, w = img.shape[:2]
        
        # Initialize gaze position if not exists (Center screen)
        if not hasattr(self, 'gaze_x'): 
            self.gaze_x, self.gaze_y = w // 2, h // 2

        # 1. Determine "Saccade Target" (Where the eye wants to go)
        # If tracking a target, look at it. If not, drift back to center.
        dest_x, dest_y = w // 2, h // 2
        if active and target_cx is not None:
            dest_x, dest_y = target_cx, target_cy

        # 2. Smooth "Eye Movement" (Inertia)
        # Eyes don't snap instantly; they zip smoothly.
        # Factor 0.15 = Fast but fluid.
        self.gaze_x = int(self.gaze_x * 0.85 + dest_x * 0.15)
        self.gaze_y = int(self.gaze_y * 0.85 + dest_y * 0.15)

        # 3. Create "Spotlight" Effect (Vignette)
        # Everything outside the gaze is slightly darker (Peripheral Vision)
        mask = np.zeros((h, w), dtype=np.uint8)
        
        # Draw white circle at gaze point (The Fovea)
        cv2.circle(mask, (self.gaze_x, self.gaze_y), 120, (255), -1)
        
        # Blur the mask heavily to create soft light falloff
        mask = cv2.GaussianBlur(mask, (99, 99), 0)
        
        # Blend: Darken the original image where mask is black
        # Convert mask to float 0..1
        mask_f = mask.astype(np.float32) / 255.0
        
        # Darken factor: 0.6 (Peripheral brightness)
        # Formula: Pixel = Pixel * (0.6 + 0.4 * mask)
        # Center stays 100% bright, edges drop to 60% bright
        img[:] = (img * (0.5 + 0.5 * mask_f[:, :, np.newaxis])).astype(np.uint8)

        # 4. Draw Gaze Markers (The "White Crosshair")
        # Shows exactly where the "Human Operator" is staring
        cv2.drawMarker(img, (self.gaze_x, self.gaze_y), (200, 200, 255), cv2.MARKER_CROSS, 20, 1)
        cv2.putText(img, "GAZE TRACK", (self.gaze_x + 15, self.gaze_y - 15), cv2.FONT_HERSHEY_PLAIN, 0.7, (180, 180, 220), 1)

    def close(self):
        if self.writer: self.writer.release()

# ==============================================================================
# 6. REPORT GENERATOR
# ==============================================================================
def generate_3d_report(vis_data, phy_data, z_data, save_path):
    print("[INFO] Generating 3D Phase Space Report...")
    fig = plt.figure(figsize=(14, 10)) # Increased width slightly
    ax = fig.add_subplot(111, projection='3d')

    # Convert to numpy for easier handling
    xs = np.array(vis_data) # Visual
    ys = np.array(phy_data) # Physics
    zs = np.array(z_data)   # Topological Risk (Z-Axis)
    
    # Create a time gradient
    t = np.arange(len(xs))

    # Scatter Plot
    sc = ax.scatter(xs, ys, zs, c=t, cmap='plasma', s=50, edgecolors='w', alpha=0.8)
    
    # Trajectory Line
    ax.plot(xs, ys, zs, color='gray', alpha=0.5, linewidth=1)

    # --- FIX: IMPROVED LABELS WITH PADDING ---
    # labelpad pushes the text away from the tick numbers
    ax.set_xlabel('Visual Anomaly (Texture)', fontweight='bold', labelpad=10)
    ax.set_ylabel('Physics Divergence (Motion)', fontweight='bold', labelpad=10)
    ax.set_zlabel('Geofence Risk (Topology)', fontweight='bold', labelpad=15) # Increased padding for Z
    
    ax.set_title('3D Decision Manifold', fontsize=16, pad=20)
    
    # Add Colorbar
    cbar = plt.colorbar(sc, pad=0.1, shrink=0.8) # shrink makes it fit better
    cbar.set_label('Frame Sequence (Time)', rotation=270, labelpad=15)

    # Aesthetics
    ax.set_facecolor('white') 
    
    # Set a better camera angle to see all labels
    ax.view_init(elev=25, azim=135) 
    
    # Save with tight layout to prevent cutting
    plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.5)
    print(f"[SUCCESS] 3D Plot saved to {save_path}")

def run_agentic_planner(alert_ctx, mood):
    v, p, f, t = alert_ctx['visual'], alert_ctx['physics'], alert_ctx['fused'], alert_ctx['topo']
    if t: return {"action": "REDIRECT", "reasoning": "Terrain Violation: Vehicle in No-Go Zone."}
    if f > 0.8: return {"action": "INTERCEPT", "reasoning": f"HYBRID THREAT: {mood} Driver + Physics Lock."}
    elif p > 0.6: return {"action": "TRACKING", "reasoning": "Physics: Non-ballistic motion."}
    elif v > 0.6: return {"action": "IDENTIFY", "reasoning": "Visual: Spectral anomaly detected."}
    else: return {"action": "PATROL", "reasoning": "Sector clear."}

# ==============================================================================
# 7. MAIN LOOP
# ==============================================================================
def main():
    cfg = Config()
    os.makedirs(cfg.out_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load Visual Model
    model = A2F_ViT().to(device)
    if os.path.exists(cfg.ckpt_path):
        print(f"[INFO] Loading trained weights from {cfg.ckpt_path}")
        model.load_state_dict(torch.load(cfg.ckpt_path, map_location=device))
    else: print("[ERROR] No weights found.")

    model.eval()
    
    # Load Data
    frames = sorted(glob.glob(os.path.join(cfg.test_folder, "**/*.jpg"), recursive=True))
    try: frames.sort(key=lambda x: int(os.path.basename(x).split('.')[0].split('_')[-1]))
    except: pass
    if not frames: print("No frames found."); return

    dash = ResearchDashboard(os.path.join(cfg.out_dir, "Human_Demo_3D.mp4"))
    dash.init_writer()
    oracle = PhysicsOracle()
    human_brain = HumanBehaviorEngine() # NEW: Initialize Human Brain
    
    print(f"[INFO] Processing {len(frames)} frames with HUMAN INTELLIGENCE...")
    
    # Calibration
    noise_samples = []
    calib_len = min(30, len(frames)-Config.num_frames)
    for i in range(calib_len):
        clip_paths = frames[i:i+Config.num_frames]
        target_path = frames[i+Config.num_frames]
        clip = []
        for p in clip_paths:
            img = cv2.imread(p)
            if img is None: continue
            img = cv2.resize(img, (224,224)).astype(np.float32)/127.5-1.0
            clip.append(torch.from_numpy(img).permute(2,0,1))
        if len(clip) < Config.num_frames: continue
        inp = torch.stack(clip).unsqueeze(0).to(device)
        t_img = cv2.resize(cv2.imread(target_path), (224,224)).astype(np.float32)/127.5-1.0
        target_ts = torch.from_numpy(t_img).permute(2,0,1).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(inp)
            raw = torch.abs(target_ts - pred).reshape(-1)
            topk, _ = torch.topk(raw, k=1000)
            noise_samples.append(topk.mean().item())
    baseline = np.mean(noise_samples) if noise_samples else 0.05
    
    # History for 3D Plot
    vis_full, phy_full, z_full = [], [], []
    
    # Main Loop
    for i in tqdm(range(0, len(frames)-Config.num_frames)):
        clip_paths = frames[i:i+Config.num_frames]
        target_path = frames[i+Config.num_frames]
        
        # Load Images
        clip = []
        for p in clip_paths:
            img = cv2.imread(p)
            if img is None: continue
            img = cv2.resize(img, (224,224)).astype(np.float32)/127.5-1.0
            clip.append(torch.from_numpy(img).permute(2,0,1))
        if len(clip) < Config.num_frames: continue
        inp = torch.stack(clip).unsqueeze(0).to(device)
        img_raw = cv2.imread(target_path)
        img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
        t_img = cv2.resize(img_raw, (224,224)).astype(np.float32)/127.5-1.0
        target_ts = torch.from_numpy(t_img).permute(2,0,1).unsqueeze(0).to(device)

        # 1. Visual
        with torch.no_grad():
            pred = model(inp)
            raw = torch.abs(target_ts - pred).reshape(-1)
            topk, _ = torch.topk(raw, k=1000)
            diff = max(0.0, topk.mean().item() - baseline)
            vis_score = min(1.0, (diff * 20.0) + 0.02)

        # 2. Physics & Topology
        bbox_norm, phy_score, topo_alert = oracle.get_state(target_path)
        
        # 3. Human Intelligence (NEW)
        driver_mood = "NORMAL"
        intent_score = 0.0
        if bbox_norm: # Only analyze if car exists
            velocity = (oracle.kf.x[2], oracle.kf.x[3])
            driver_mood, intent_score = human_brain.analyze_intent(velocity)
        
        # 4. Fusion
        human_penalty = 0.2 if driver_mood == "AGGRESSIVE" else 0.0
        fused = (0.5 * vis_score) + (0.3 * phy_score) + (0.2 * intent_score) + human_penalty
        is_alert = (fused > Config.planner_trigger_score) or topo_alert
        
        # Store for 3D Plot
        topo_risk = 0.0
        if bbox_norm:
            cx, cy = bbox_norm[0] + bbox_norm[2]/2, bbox_norm[1] + bbox_norm[3]/2
            dist = np.sqrt((cx - 0.5)**2 + (cy - 0.5)**2)
            topo_risk = max(0.0, 1.0 - (dist * 2.0))
        vis_full.append(vis_score); phy_full.append(phy_score); z_full.append(topo_risk)

        # Update Dash
        ctx = {'visual': vis_score, 'physics': phy_score, 'fused': fused, 'topo': topo_alert}
        plan = run_agentic_planner(ctx, driver_mood)
        state = {'vis': vis_score, 'phy': phy_score, 'alert': is_alert, 'topo': topo_alert}
        dash.update(img_rgb, bbox_norm, state, plan, driver_mood)

    dash.close()
    generate_3d_report(vis_full, phy_full, z_full, os.path.join(cfg.out_dir, "3D_Analysis_3D.png"))
    print(f"[SUCCESS] Saved Video: {dash.save_path}")

if __name__ == "__main__":
    main()

[SUCCESS] Gemini API Key loaded from Secrets.
[INFO] Loading trained weights from /kaggle/input/datasets/amankumarpatell/da-br-path/best_da_vit_BR.pth
[INFO] Processing 5700 frames with HUMAN INTELLIGENCE...


100%|██████████| 5696/5696 [16:51<00:00,  5.63it/s]


[INFO] Generating 3D Phase Space Report...
[SUCCESS] 3D Plot saved to /kaggle/working/3D_Analysis_3D.png
[SUCCESS] Saved Video: /kaggle/working/Human_Demo_3D.mp4


In [4]:
# ==============================================================================
# A3F-ViT: GENESIS-RAPI (COMPLETE UNIFIED CORE)
# Preserving: Spectral Signal, Kalman Physics, Human Jerk, Geofence, 3D Report.
# Patching: Active Inference (VFE Surprise) & Forensic RAG Reasoning.
# ==============================================================================

import os
import glob
import numpy as np
import cv2
from tqdm import tqdm
from collections import deque
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from timm.models.vision_transformer import Attention

# --- SYSTEM MODULES ---
try:
    from ultralytics import YOLO
    from filterpy.kalman import KalmanFilter
    HAS_PHYSICS = True
except ImportError:
    HAS_PHYSICS = False

# ==============================================================================
# 1. THE GENESIS PATCH: COGNITIVE & FORENSIC MODULES
# ==============================================================================

class GenesisWorldModel(nn.Module):
    """LSTM-based latent rehearsal for Variational Free Energy."""
    def __init__(self, latent_dim=768):
        super().__init__()
        self.transition = nn.LSTM(latent_dim, latent_dim, batch_first=True)
    def forward(self, x, hidden=None):
        return self.transition(x, hidden)

class ForensicRAG:
    """Retrieves legal context to justify intercepts."""
    def __init__(self):
        self.kb = {
            "ZONE_VIOLATION": "FAA Sec 14: Unauthorized entry into Pedestrian Safety Zone.",
            "KINEMATIC_DANGER": "Physics Law 4: Non-ballistic erratic motion detected.",
            "VISUAL_SURPRISE": "Incident Protocol 3: High reconstruction error (Anomalous Texture)."
        }
    def justify(self, alert_type, score):
        rule = self.kb.get(alert_type, "Standard Observation Protocol.")
        return f"JUSTIFICATION: {rule} (VFE: {score:.2f})"

# ==============================================================================
# 2. CONFIGURATION & ORIGINAL ARCHITECTURE
# ==============================================================================

class Config:
    exp_name = "A3F_ViT_Genesis_Unified"
    out_dir = "/kaggle/working/"
    test_folder = "/kaggle/input/datasets/amankumarpatell/drone-anomaly/Bike Roundabout/sequence1/test/04"
    ckpt_path   = "/kaggle/input/datasets/amankumarpatell/da-br-path/best_da_vit_BR.pth"
    video_fps = 15.0
    canvas_size = (1280, 720)
    image_size = 224
    num_frames = 4
    planner_trigger_score = 0.60 

class RadialMaskMLP(nn.Module):
    def __init__(self, hidden_dim=32, layers=3):
        super().__init__()
        seq = [nn.Linear(1, hidden_dim), nn.ReLU()]
        for _ in range(layers-1):
            seq.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        seq.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*seq)
    def forward(self, r):
        w = F.softplus(self.net(r.view(-1, 1)))
        return w.view(r.shape) / (w.max() + 1e-6)

class AttentionWithMap(Attention):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_map = None
    def forward(self, x, attn_mask=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        self.attn_map = attn 
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class A2F_ViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model('vit_base_patch16_224', pretrained=False)
        for blk in self.encoder.blocks:
            old = blk.attn
            blk.attn = AttentionWithMap(dim=768, num_heads=old.num_heads, qkv_bias=True)
        self.decoder = nn.Sequential(
            nn.Conv2d(768 * 4, 512, 1),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64, 3, 3, 2, 1, 1), nn.Tanh()
        )
        self.radial_mlp = RadialMaskMLP()
    def forward(self, x):
        B, T = x.shape[:2]
        feats = []
        for i in range(T):
            f = self.encoder.forward_features(x[:, i])[:, 1:]
            f = f.permute(0, 2, 1).reshape(B, -1, 14, 14)
            feats.append(f)
        return self.decoder(torch.cat(feats, 1))

class HumanBehaviorEngine:
    def __init__(self):
        self.vel_history = deque(maxlen=10)
    def analyze_intent(self, vel):
        self.vel_history.append(vel)
        if len(self.vel_history) < 3: return "ANALYZING", 0.0
        accels = [np.linalg.norm(np.array(self.vel_history[i]) - np.array(self.vel_history[i-1])) for i in range(1, len(self.vel_history))]
        avg_jerk = np.mean(np.abs(np.diff(accels))) * 100.0
        score = min(1.0, avg_jerk / 5.0)
        return ("AGGRESSIVE" if score > 0.8 else "DISTRACTED" if score > 0.4 else "NORMAL"), score

class PhysicsOracle:
    def __init__(self):
        if HAS_PHYSICS:
            self.model = YOLO('yolov8n.pt')
            self.kf = KalmanFilter(dim_x=4, dim_z=2)
            self.kf.F = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]])
            self.kf.H = np.array([[1,0,0,0],[0,1,0,0]])
            self.kf.P *= 1000.; self.kf.R *= 5.; self.kf.Q *= 5.0
            self.kf.x = np.zeros(4)
    def get_state(self, path):
        if not HAS_PHYSICS: return None, 0.0, False
        res = self.model(path, classes=[2,3,5,7], verbose=False)
        bbox, score, topo = None, 0.0, False
        if len(res[0].boxes) > 0:
            b = res[0].boxes.xywh.cpu().numpy()[0]
            H, W = res[0].orig_shape
            bbox = (b[0]/W, b[1]/H, b[2]/W, b[3]/H)
            dist = np.sqrt((bbox[0]-0.5)**2 + (bbox[1]-0.5)**2)
            if dist < 0.12 or dist > 0.48: topo = True
            self.kf.predict(); self.kf.update(np.array([bbox[0], bbox[1]]))
            score = np.linalg.norm(np.array([bbox[0], bbox[1]]) - (self.kf.H @ self.kf.x_prior)) * 10.0
        return bbox, min(1.0, score), topo

# ==============================================================================
# 3. RESEARCH DASHBOARD (WITH GENESIS PATCHED METHODS)
# ==============================================================================

class ResearchDashboard:
    def __init__(self, save_path):
        self.save_path = save_path
        self.writer = None
        self.log_trace = deque(["Genesis-RAPI Initialized."], maxlen=8)
        self.vis_trace = deque([0.0]*50, maxlen=50)
        self.phy_trace = deque([0.0]*50, maxlen=50)
        self.smooth_vis, self.smooth_phy = 0.0, 0.0
        self.lock_timer = 0
        self.trajectory = deque(maxlen=20); self.radar_trail = deque(maxlen=20)
        self.c_bg = (10, 15, 20); self.c_panel = (30, 35, 40)
        self.c_vis = (0, 165, 255); self.c_phy = (255, 200, 0); self.c_crit = (0, 0, 255); self.c_topo = (255, 0, 255)

    def init_writer(self):
        self.writer = cv2.VideoWriter(self.save_path, cv2.VideoWriter_fourcc(*'mp4v'), Config.video_fps, Config.canvas_size)

    # [INSTRUCTIONS: ADD YOUR ORIGINAL _draw_phase_radar, _draw_tactical_hud, _draw_bio_gaze HERE]
    # (Simplified placeholders below to maintain functionality in one cell)
    def _draw_hud(self, feed, bbox, state):
        if bbox:
            cx, cy = int(bbox[0]*720), int(bbox[1]*720)
            cv2.drawMarker(feed, (cx, cy), self.c_vis, cv2.MARKER_CROSS, 20, 2)

    def update(self, frame_rgb, bbox, state, plan, mood):
        canvas = np.zeros((720, 1280, 3), dtype=np.uint8); canvas[:] = self.c_bg
        self.smooth_vis = 0.8*self.smooth_vis + 0.2*state['vis']
        self.smooth_phy = 0.8*self.smooth_phy + 0.2*state['phy']
        self.vis_trace.append(self.smooth_vis); self.phy_trace.append(self.smooth_phy)

        if state['alert']:
            msg = f"> {plan['reasoning'][:55]}"
            if self.log_trace[-1] != msg: self.log_trace.append(msg)

        feed = cv2.resize(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR), (720, 720))
        self._draw_hud(feed, bbox, state)
        
        cv2.rectangle(canvas, (740, 550), (1250, 700), self.c_panel, -1)
        for i, log in enumerate(self.log_trace):
            col = (200,200,200) if "JUSTIFICATION" not in log else (0, 255, 255)
            cv2.putText(canvas, log, (750, 580 + (i*15)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1)
        
        canvas[0:720, 0:720] = feed
        if self.writer: self.writer.write(canvas)

# ==============================================================================
# 4. MAIN LOOP (EXECUTION ON 2353 FRAMES)
# ==============================================================================

def main():
    cfg = Config(); device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = A2F_ViT().to(device); world = GenesisWorldModel().to(device); rag = ForensicRAG()
    oracle = PhysicsOracle(); behavior = HumanBehaviorEngine()
    
    if os.path.exists(cfg.ckpt_path): model.load_state_dict(torch.load(cfg.ckpt_path, map_location=device), strict=False)

    dash = ResearchDashboard(os.path.join(cfg.out_dir, "Genesis_Integrated_Defense.mp4")); dash.init_writer()
    frames = sorted(glob.glob(os.path.join(cfg.test_folder, "*.jpg")))
    hidden = None

    for i in tqdm(range(len(frames)-cfg.num_frames)):
        target_path = frames[i+cfg.num_frames]
        
        # --- NEW CONCEPT PATCH: LSTM-FLATTENED SURPRISE ---
        with torch.no_grad():
            img = cv2.resize(cv2.imread(target_path), (224,224)).astype(np.float32)/127.5-1.0
            # Get ViT Features: (1, 196, 768)
            feat = model.encoder.forward_features(torch.from_numpy(img).permute(2,0,1).unsqueeze(0).to(device))[:, 1:]
            
            # PATCH: Mean-pool spatial tokens to get global feature (1, 1, 768)
            global_feat = feat.mean(dim=1, keepdim=True) 
            
            # Predict next state using LSTM
            pred_feat, hidden = world(global_feat, hidden)
            
            # VFE Surprise Calculation
            vis_score = min(1.0, torch.norm(global_feat - pred_feat).item() / 8.0)

        # --- PRESERVING ORIGINAL LOGIC ---
        bbox, phy_score, topo = oracle.get_state(target_path)
        mood, jerk = behavior.analyze_intent(oracle.kf.x[2:4]) if bbox else ("NORMAL", 0.0)
        
        alert_type = "ZONE_VIOLATION" if topo else "KINEMATIC_DANGER" if phy_score > vis_score else "VISUAL_SURPRISE"
        plan = {"reasoning": rag.justify(alert_type, (vis_score + phy_score)/2)}
        state = {'vis': vis_score, 'phy': phy_score, 'alert': (vis_score+phy_score)/2 > 0.5, 'topo': topo}
        
        dash.update(cv2.imread(target_path), bbox, state, plan, mood)
    
    dash.writer.release()
    print(f"[SUCCESS] Video saved to {dash.save_path}")

if __name__ == "__main__":
    main()

  1%|          | 50/5696 [00:03<07:01, 13.38it/s] 


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# A3F-ViT: TOPOLOGY-AWARE NEURO-SYMBOLIC CORE (WITH HUMAN INTELLIGENCE)
# "FINAL DEFENSE EDITION" - DEC 4TH TARGET
# Includes: Spectral Visual, Kalman Physics, Human Behavior (Jerk), Geofence, & 3D Report
# ==============================================================================

import os
import glob
import numpy as np
import cv2
from tqdm import tqdm
from collections import deque
import matplotlib
matplotlib.use('Agg') # Backend for saving plots without display
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from timm.models.vision_transformer import Attention

# --- SYSTEM MODULES ---
try:
    from ultralytics import YOLO
    from filterpy.kalman import KalmanFilter
    HAS_PHYSICS = True
except ImportError:
    print("[WARN] YOLO or FilterPy not found. Physics module will be disabled.")
    HAS_PHYSICS = False

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
class Config:
    exp_name = "A3F_ViT_Human"
    out_dir = "/kaggle/working/"
    # UPDATE THESE PATHS BEFORE RUNNING
    test_folder = "/kaggle/input/dataset1/archive/50m_90d_morning_congkhuA_22_3_test"
    ckpt_path   = "/kaggle/input/visualpath/visual_backbone.pth"
    
    video_fps = 15.0
    canvas_size = (1280, 720)
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        gemini_api_key = user_secrets.get_secret("AAAW_SECRET_KEY")
        print("[SUCCESS] Gemini API Key loaded from Secrets.")
    except Exception as e:
        print(f"[WARN] Could not load API Key: {e}. Using Mock.")
        gemini_api_key = "MOCK_KEY"
        
    image_size = 224
    num_frames = 4
    planner_trigger_score = 0.60 

# ==============================================================================
# 2. MODEL DEFINITION (Visual Cortex)
# ==============================================================================
class RadialMaskMLP(nn.Module):
    def __init__(self, hidden_dim=32, layers=3):
        super().__init__()
        seq = [nn.Linear(1, hidden_dim), nn.ReLU()]
        for _ in range(layers-1):
            seq.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        seq.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*seq)
    def forward(self, r):
        w = F.softplus(self.net(r.view(-1, 1)))
        return w.view(r.shape) / (w.max() + 1e-6)

class AttentionWithMap(Attention):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_map = None
    def forward(self, x, attn_mask=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        self.attn_map = attn 
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class A2F_ViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model('vit_base_patch16_224', pretrained=False)
        for blk in self.encoder.blocks:
            old = blk.attn
            new_attn = AttentionWithMap(dim=768, num_heads=old.num_heads, qkv_bias=True)
            blk.attn = new_attn
        self.decoder = nn.Sequential(
            nn.Conv2d(768 * 4, 512, 1),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64, 3, 3, 2, 1, 1), nn.Tanh()
        )
        self.radial_mlp = RadialMaskMLP()

    def forward(self, x):
        B, T = x.shape[:2]
        feats = []
        for i in range(T):
            f = self.encoder.forward_features(x[:, i])[:, 1:]
            f = f.permute(0, 2, 1).reshape(B, -1, 14, 14)
            feats.append(f)
        return self.decoder(torch.cat(feats, 1))

# ==============================================================================
# 3. HUMAN INTELLIGENCE ENGINE (NEW)
# ==============================================================================
class HumanBehaviorEngine:
    def __init__(self):
        # Store history of velocities to calculate acceleration/jerk
        self.vel_history = deque(maxlen=10)
    
    def analyze_intent(self, velocity_vector):
        """
        Infers driver state based on motion smoothness (Jerk).
        Input: velocity_vector (vx, vy)
        Returns: State String, Intent Score (0.0 - 1.0)
        """
        self.vel_history.append(velocity_vector)
        
        # Need at least 3 frames to calculate Jerk (Change in Accel)
        if len(self.vel_history) < 3:
            return "ANALYZING", 0.0

        # 1. Calculate Acceleration (Delta V)
        accels = []
        for i in range(1, len(self.vel_history)):
            v1 = np.array(self.vel_history[i-1])
            v2 = np.array(self.vel_history[i])
            accels.append(np.linalg.norm(v2 - v1))
            
        # 2. Calculate Jerk (Delta A) - The "Comfort" Metric
        if len(accels) < 2: return "ANALYZING", 0.0
        
        jerks = np.diff(accels)
        avg_jerk = np.mean(np.abs(jerks)) * 100.0 # Scale up for readability
        
        # 3. Classify "Human State"
        intent_score = min(1.0, avg_jerk / 5.0) # Normalize 0-1
        
        if intent_score > 0.8: return "AGGRESSIVE", intent_score
        elif intent_score > 0.4: return "DISTRACTED", intent_score
        elif intent_score < 0.1: return "CAUTIOUS", intent_score
        else: return "NORMAL", intent_score

# ==============================================================================
# 4. PHYSICS ORACLE
# ==============================================================================
class PhysicsOracle:
    def __init__(self):
        if HAS_PHYSICS:
            try: self.model = YOLO('yolov8n.pt')
            except: self.model = None
            
            # Kalman Filter Setup
            self.kf = KalmanFilter(dim_x=4, dim_z=2) # [x, y, vx, vy]
            self.kf.F = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]])
            self.kf.H = np.array([[1,0,0,0],[0,1,0,0]])
            self.kf.P *= 1000.; self.kf.R *= 5.; self.kf.Q *= 5.0
            self.kf.x = np.zeros(4)
            
    def get_state(self, frame_path):
        if not HAS_PHYSICS or self.model is None:
            return None, 0.0, False

        results = self.model(frame_path, classes=[2,3,5,7], verbose=False)
        bbox_norm = None
        score = 0.0
        topo_alert = False 
        
        if len(results[0].boxes) > 0:
            b = results[0].boxes.xywh.cpu().numpy()[0]
            H, W = results[0].orig_shape
            bbox_norm = (b[0]/W, b[1]/H, b[2]/W, b[3]/H) 
            
            # Topology Check
            dist_from_center = np.sqrt((bbox_norm[0]-0.5)**2 + (bbox_norm[1]-0.5)**2)
            if dist_from_center < 0.12 or dist_from_center > 0.48:
                topo_alert = True
            
            # Kalman Update
            z = np.array([bbox_norm[0], bbox_norm[1]])
            self.kf.predict()
            self.kf.update(z)
            pred = self.kf.H @ self.kf.x_prior
            score = np.linalg.norm(z - pred) * 10.0 
            
        return bbox_norm, min(1.0, score), topo_alert

# ==============================================================================
# 5. DASHBOARD & VISUALIZATION
# ==============================================================================
class ResearchDashboard:
    def __init__(self, save_path):
        self.save_path = save_path
        self.writer = None
        self.history_len = 50
        self.vis_trace = deque([0.0]*self.history_len, maxlen=self.history_len)
        self.phy_trace = deque([0.0]*self.history_len, maxlen=self.history_len)
        self.log_trace = deque(["System Initialized...", "Human Engine Active."], maxlen=8)
        self.trajectory = deque(maxlen=20) 
        self.radar_trail = deque(maxlen=20) 
        self.smooth_vis = 0.0
        self.smooth_phy = 0.0
        self.lock_timer = 0  
        
        self.c_bg = (10, 15, 20); self.c_panel = (30, 35, 40)
        self.c_vis = (0, 165, 255); self.c_phy = (255, 200, 0)
        self.c_crit = (0, 0, 255); self.c_scan = (255, 255, 0) 
        self.c_safe = (0, 255, 0); self.c_topo = (255, 0, 255) 
        self.c_txt = (220, 220, 220)
        
    def init_writer(self):
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        self.writer = cv2.VideoWriter(self.save_path, fourcc, Config.video_fps, Config.canvas_size)

    def _draw_phase_radar(self, img, state_vec, pos, size=180):
        x0, y0 = pos
        cx, cy = x0 + size//2, y0 + size//2
        radius = size // 2 - 10
        cv2.circle(img, (cx, cy), radius, (50, 50, 50), 1)
        cv2.line(img, (cx, y0+10), (cx, y0+size-10), (50, 50, 50), 1)
        cv2.line(img, (x0+10, cy), (x0+size-10, cy), (50, 50, 50), 1)
        cv2.putText(img, "PHY", (x0+5, y0+20), cv2.FONT_HERSHEY_SIMPLEX, 0.4, self.c_phy, 1)
        cv2.putText(img, "VIS", (x0+size-30, y0+size-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, self.c_vis, 1)
        
        jitter_x = np.random.randint(-1, 2); jitter_y = np.random.randint(-1, 2)
        px = int(cx + (state_vec['vis'] * radius)) + jitter_x
        py = int(cy - (state_vec['phy'] * radius)) + jitter_y
        
        self.radar_trail.append((px, py))
        for i in range(1, len(self.radar_trail)):
            thick = max(1, int(2 * i/len(self.radar_trail)))
            cv2.line(img, self.radar_trail[i-1], self.radar_trail[i], (200, 200, 200), thick)
        
        col = self.c_scan
        if state_vec['alert']: col = self.c_crit
        elif state_vec['topo']: col = self.c_topo
        cv2.circle(img, (px, py), 6, col, -1)
        cv2.circle(img, (px, py), 8, (255, 255, 255), 1)

    def _draw_graph(self, img, roi, data, color, label):
        x0, y0, w, h = roi
        cv2.rectangle(img, (x0, y0), (x0+w, y0+h), self.c_panel, -1)
        cv2.putText(img, label, (x0+10, y0+20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, self.c_txt, 1)
        if len(data) < 2: return
        pts = []
        max_val = max(1.0, max(data))
        for i, val in enumerate(data):
            px = int(x0 + (i / (self.history_len-1)) * w)
            py = int(y0 + h - (val / max_val) * (h-10) - 5)
            pts.append((px, py))
        for i in range(1, len(pts)): cv2.line(img, pts[i-1], pts[i], color, 2)
        cv2.putText(img, f"{data[-1]:.2f}", (x0+w-50, y0+20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    def update(self, frame_rgb, bbox_norm, state, plan, mood):
        canvas = np.zeros((Config.canvas_size[1], Config.canvas_size[0], 3), dtype=np.uint8)
        canvas[:] = self.c_bg
        
        self.smooth_vis = (0.85 * self.smooth_vis) + (0.15 * state['vis'])
        self.smooth_phy = (0.85 * self.smooth_phy) + (0.15 * state['phy'])
        self.vis_trace.append(self.smooth_vis)
        self.phy_trace.append(self.smooth_phy)
        
        if plan and plan.get('reasoning') and plan['reasoning'] != self.log_trace[-1]:
            self.log_trace.append(f"> {plan['action']}: {plan['reasoning'][:30]}...")

        # Feed
        feed_size = 720
        feed = cv2.resize(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR), (feed_size, feed_size))
        center_x, center_y = feed_size // 2, feed_size // 2
        
        # Geofence Rings
        ring_color = (60, 20, 60); thickness = 1
        if state['topo']: ring_color = (255, 0, 255); thickness = 3
        cv2.circle(feed, (center_x, center_y), int(0.12*feed_size*2), ring_color, thickness)
        cv2.circle(feed, (center_x, center_y), int(0.48*feed_size*2), ring_color, thickness)
        
        # Bounding Box
        if bbox_norm:
            nx, ny, nw, nh = bbox_norm
            cx, cy = int(nx*feed_size), int(ny*feed_size)
            if len(self.trajectory) > 0:
                last_cx, last_cy = self.trajectory[-1]
                if np.sqrt((cx-last_cx)**2 + (cy-last_cy)**2) > 100: self.trajectory.clear()
            self.trajectory.append((cx, cy))
            
            color = self.c_scan; status = "SCANNING"
            if state['alert']: self.lock_timer = 30
            if self.lock_timer > 0: color = self.c_crit; status = "HYBRID LOCK"; self.lock_timer -= 1
            elif state['topo']: color = self.c_topo; status = "GEOFENCE BREACH"
            
            bw, bh = int(nw*feed_size), int(nh*feed_size)
            cv2.rectangle(feed, (cx-bw//2, cy-bh//2), (cx+bw//2, cy+bh//2), color, 2)
            cv2.putText(feed, f"{status} | {mood}", (cx-bw//2, cy-bh//2-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
            for i in range(1, len(self.trajectory)):
                thick = max(1, int(3 * i/len(self.trajectory)))
                cv2.line(feed, self.trajectory[i-1], self.trajectory[i], color, thick)
        else:
            self.trajectory.clear()
        
        canvas[0:720, 0:720] = feed
        
        # Panel
        panel_x = 740
        cv2.putText(canvas, " NEURO-SYMBOLIC CORE", (panel_x, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        smooth_state = {'vis': self.smooth_vis, 'phy': self.smooth_phy, 'alert': (self.lock_timer > 0), 'topo': state['topo']}
        
        self._draw_phase_radar(canvas, smooth_state, (panel_x + 150, 60), size=200)
        self._draw_graph(canvas, (panel_x, 300, 500, 100), self.vis_trace, self.c_vis, "SPECTRAL RESIDUAL (REAL)")
        self._draw_graph(canvas, (panel_x, 420, 500, 100), self.phy_trace, self.c_phy, "KALMAN DIVERGENCE (REAL)")
        
        # Human Behavior Display
        cv2.putText(canvas, f"DRIVER PROFILE: {mood}", (panel_x+10, 540), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        cv2.rectangle(canvas, (panel_x, 550), (panel_x+500, 700), self.c_panel, -1)
        cv2.putText(canvas, "AGENTIC THOUGHT STREAM:", (panel_x+10, 575), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)
        for i, log in enumerate(self.log_trace):
            col = self.c_txt
            if "INTERCEPT" in log: col = self.c_crit
            elif "Violation" in log: col = self.c_topo 
            cv2.putText(canvas, log, (panel_x+10, 600 + (i*20)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
            
        if self.writer: self.writer.write(canvas)

    def close(self):
        if self.writer: self.writer.release()

# ==============================================================================
# 6. REPORT GENERATOR
# ==============================================================================
def generate_3d_report(vis_data, phy_data, z_data, save_path):
    print("[INFO] Generating 3D Phase Space Report...")
    fig = plt.figure(figsize=(14, 10)) # Increased width slightly
    ax = fig.add_subplot(111, projection='3d')

    # Convert to numpy for easier handling
    xs = np.array(vis_data) # Visual
    ys = np.array(phy_data) # Physics
    zs = np.array(z_data)   # Topological Risk (Z-Axis)
    
    # Create a time gradient
    t = np.arange(len(xs))

    # Scatter Plot
    sc = ax.scatter(xs, ys, zs, c=t, cmap='plasma', s=50, edgecolors='w', alpha=0.8)
    
    # Trajectory Line
    ax.plot(xs, ys, zs, color='gray', alpha=0.5, linewidth=1)

    # --- FIX: IMPROVED LABELS WITH PADDING ---
    # labelpad pushes the text away from the tick numbers
    ax.set_xlabel('Visual Anomaly (Texture)', fontweight='bold', labelpad=10)
    ax.set_ylabel('Physics Divergence (Motion)', fontweight='bold', labelpad=10)
    ax.set_zlabel('Geofence Risk (Topology)', fontweight='bold', labelpad=15) # Increased padding for Z
    
    ax.set_title('A3F-ViT 3D Decision Manifold', fontsize=16, pad=20)
    
    # Add Colorbar
    cbar = plt.colorbar(sc, pad=0.1, shrink=0.8) # shrink makes it fit better
    cbar.set_label('Frame Sequence (Time)', rotation=270, labelpad=15)

    # Aesthetics
    ax.set_facecolor('white') 
    
    # Set a better camera angle to see all labels
    ax.view_init(elev=25, azim=135) 
    
    # Save with tight layout to prevent cutting
    plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.5)
    print(f"[SUCCESS] 3D Plot saved to {save_path}")

def run_agentic_planner(alert_ctx, mood):
    v, p, f, t = alert_ctx['visual'], alert_ctx['physics'], alert_ctx['fused'], alert_ctx['topo']
    if t: return {"action": "REDIRECT", "reasoning": "Terrain Violation: Vehicle in No-Go Zone."}
    if f > 0.8: return {"action": "INTERCEPT", "reasoning": f"HYBRID THREAT: {mood} Driver + Physics Lock."}
    elif p > 0.6: return {"action": "TRACKING", "reasoning": "Physics: Non-ballistic motion."}
    elif v > 0.6: return {"action": "IDENTIFY", "reasoning": "Visual: Spectral anomaly detected."}
    else: return {"action": "PATROL", "reasoning": "Sector clear."}

# ==============================================================================
# 7. MAIN LOOP
# ==============================================================================
def main():
    cfg = Config()
    os.makedirs(cfg.out_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load Visual Model
    model = A2F_ViT().to(device)
    if os.path.exists(cfg.ckpt_path):
        print(f"[INFO] Loading trained weights from {cfg.ckpt_path}")
        model.load_state_dict(torch.load(cfg.ckpt_path, map_location=device))
    else: print("[ERROR] No weights found.")

    model.eval()
    
    # Load Data
    frames = sorted(glob.glob(os.path.join(cfg.test_folder, "**/*.jpg"), recursive=True))
    try: frames.sort(key=lambda x: int(os.path.basename(x).split('.')[0].split('_')[-1]))
    except: pass
    if not frames: print("No frames found."); return

    dash = ResearchDashboard(os.path.join(cfg.out_dir, "Human_Demo.mp4"))
    dash.init_writer()
    oracle = PhysicsOracle()
    human_brain = HumanBehaviorEngine() # NEW: Initialize Human Brain
    
    print(f"[INFO] Processing {len(frames)} frames with HUMAN INTELLIGENCE...")
    
    # Calibration
    noise_samples = []
    calib_len = min(30, len(frames)-Config.num_frames)
    for i in range(calib_len):
        clip_paths = frames[i:i+Config.num_frames]
        target_path = frames[i+Config.num_frames]
        clip = []
        for p in clip_paths:
            img = cv2.imread(p)
            if img is None: continue
            img = cv2.resize(img, (224,224)).astype(np.float32)/127.5-1.0
            clip.append(torch.from_numpy(img).permute(2,0,1))
        if len(clip) < Config.num_frames: continue
        inp = torch.stack(clip).unsqueeze(0).to(device)
        t_img = cv2.resize(cv2.imread(target_path), (224,224)).astype(np.float32)/127.5-1.0
        target_ts = torch.from_numpy(t_img).permute(2,0,1).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(inp)
            raw = torch.abs(target_ts - pred).reshape(-1)
            topk, _ = torch.topk(raw, k=1000)
            noise_samples.append(topk.mean().item())
    baseline = np.mean(noise_samples) if noise_samples else 0.05
    
    # History for 3D Plot
    vis_full, phy_full, z_full = [], [], []
    
    # Main Loop
    for i in tqdm(range(0, len(frames)-Config.num_frames)):
        clip_paths = frames[i:i+Config.num_frames]
        target_path = frames[i+Config.num_frames]
        
        # Load Images
        clip = []
        for p in clip_paths:
            img = cv2.imread(p)
            if img is None: continue
            img = cv2.resize(img, (224,224)).astype(np.float32)/127.5-1.0
            clip.append(torch.from_numpy(img).permute(2,0,1))
        if len(clip) < Config.num_frames: continue
        inp = torch.stack(clip).unsqueeze(0).to(device)
        img_raw = cv2.imread(target_path)
        img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
        t_img = cv2.resize(img_raw, (224,224)).astype(np.float32)/127.5-1.0
        target_ts = torch.from_numpy(t_img).permute(2,0,1).unsqueeze(0).to(device)

        # 1. Visual
        with torch.no_grad():
            pred = model(inp)
            raw = torch.abs(target_ts - pred).reshape(-1)
            topk, _ = torch.topk(raw, k=1000)
            diff = max(0.0, topk.mean().item() - baseline)
            vis_score = min(1.0, (diff * 20.0) + 0.02)

        # 2. Physics & Topology
        bbox_norm, phy_score, topo_alert = oracle.get_state(target_path)
        
        # 3. Human Intelligence (NEW)
        driver_mood = "NORMAL"
        intent_score = 0.0
        if bbox_norm: # Only analyze if car exists
            velocity = (oracle.kf.x[2], oracle.kf.x[3])
            driver_mood, intent_score = human_brain.analyze_intent(velocity)
        
        # 4. Fusion
        human_penalty = 0.2 if driver_mood == "AGGRESSIVE" else 0.0
        fused = (0.5 * vis_score) + (0.3 * phy_score) + (0.2 * intent_score) + human_penalty
        is_alert = (fused > Config.planner_trigger_score) or topo_alert
        
        # Store for 3D Plot
        topo_risk = 0.0
        if bbox_norm:
            cx, cy = bbox_norm[0] + bbox_norm[2]/2, bbox_norm[1] + bbox_norm[3]/2
            dist = np.sqrt((cx - 0.5)**2 + (cy - 0.5)**2)
            topo_risk = max(0.0, 1.0 - (dist * 2.0))
        vis_full.append(vis_score); phy_full.append(phy_score); z_full.append(topo_risk)

        # Update Dash
        ctx = {'visual': vis_score, 'physics': phy_score, 'fused': fused, 'topo': topo_alert}
        plan = run_agentic_planner(ctx, driver_mood)
        state = {'vis': vis_score, 'phy': phy_score, 'alert': is_alert, 'topo': topo_alert}
        dash.update(img_rgb, bbox_norm, state, plan, driver_mood)

    dash.close()
    generate_3d_report(vis_full, phy_full, z_full, os.path.join(cfg.out_dir, "3D_Analysis.png"))
    print(f"[SUCCESS] Saved Video: {dash.save_path}")

if __name__ == "__main__":
    main()